# 365 Probabilidades - Dia #156
## Qual a probabilidade de dormir até tarde no sábado compensar a semana?

**Tipo:** Saúde · Sono
**Data de publicação:** 2026-11-16
**Ferramenta:** Python
**Decisão analisada:** Vale dormir pouco na semana contando com o fim de semana para recuperar?
**Hashtag:** #365Probabilidades #Dia156

---

### 📖 A História

Todo mundo já fez essa promessa. A semana aperta, o sono fica para depois, e o plano é
simples: no sábado eu durmo até tarde e recupero tudo.

O despertador não toca, o corpo agradece, e a segunda chega com a sensação de conta paga.
A pergunta deste dia é se essa conta fecha mesmo.

---

### 📚 O Conceito: a dívida de sono

Quando a pessoa dorme menos do que precisa, a diferença vira uma espécie de dívida. O corpo
sente na atenção, no humor e também no metabolismo, no jeito como lida com o açúcar no sangue.

A ideia de pagar essa dívida no fim de semana supõe que o sono funciona como uma conta
bancária: tirou na semana, deposita no sábado. O problema é que o relógio do corpo também
entra na conta. Dormir até tarde no sábado empurra o relógio para frente, e a segunda-feira
chega com o corpo num fuso diferente. Till Roenneberg chamou isso de jet lag social.

---

### 🧮 O Modelo

**Fontes:**
- Depner, C. M., Melanson, E. L., Eckel, R. H., Snell-Bergeon, J. K., Perreault, L., Bergman,
  B. C., Higgins, J. A., Guerin, M. K., Stothard, E. R., Morton, S. J. & Wright, K. P. Jr.
  (2019). Ad libitum weekend recovery sleep fails to prevent metabolic dysregulation during a
  repeating pattern of insufficient sleep and weekend recovery sleep. *Current Biology*, 29(6),
  957-967. Universidade do Colorado. **36 adultos saudáveis**, duas semanas morando num
  laboratório do sono, sorteados em três grupos: até 9 horas por noite; no máximo 5 horas o
  tempo todo; 5 horas por 5 dias, sono livre por 2 dias e de novo 5 horas. Sensibilidade à
  insulina: **−13%** no grupo de sono curto e **−27%** no grupo que recuperou no fim de
  semana. No fim de semana, o grupo da recuperação dormiu cerca de uma hora a mais por noite.
  Números do resumo e do comunicado do NIH; os desvios por grupo não foram lidos no texto
  completo.
- Åkerstedt, T., Ghilotti, F., Grotta, A., Zhao, H., Adami, H.-O., Trolle-Lagerros, Y. &
  Bellocco, R. (2019). Sleep duration and mortality: Does weekend sleep matter? *Journal of
  Sleep Research*, 28(1), e12712. Suécia, coorte de **43.880 pessoas** (38.015 analisadas),
  13 anos, 3.234 mortes. Tabelas 1 e 2 (pessoas, mortes e pessoas-ano por grupo) lidas no
  texto completo. Abaixo de 65 anos, ajustado por sexo, IMC, escolaridade, fumo, álcool,
  atividade física, trabalho em turnos e doenças: dormir 5 horas ou menos a semana inteira,
  HR **1,65 [1,22; 2,23]**; dormir 8 horas ou mais a semana inteira, HR **1,25 [1,05; 1,50]**;
  dormir pouco na semana e mais no fim de semana, sem diferença do grupo de 6 a 7 horas (o
  número exato está só num gráfico). Acima de 65 anos, nenhuma associação.
- Roenneberg, T., Allebrandt, K. V., Merrow, M. & Vetter, C. (2012). Social jetlag and obesity.
  *Current Biology*, 22(10), 939-943. Conceito de jet lag social, ligado a mais peso mesmo
  descontando as horas de sono. Sem número no modelo.

**Desenho e família do teste.**
- Camada 1: contabilidade da dívida de sono, conta própria com premissas declaradas e
  análise de sensibilidade.
- Camada 2: o metabolismo no experimento, comparação das quedas publicadas.
- Camada 3: taxas brutas de morte por 1.000 pessoas-ano em cada grupo da coorte sueca, com
  posterior Gama (prior de Jeffreys) e Monte Carlo de 200 mil sorteios.
- Camada 4: as razões de risco ajustadas publicadas, com IC 95%, na escala log.

**Sem ×0,80.** São medidas de laboratório e mortes registradas. As horas de sono da coorte
sueca são autorrelatadas, mas são a exposição, não uma proporção a corrigir.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 156
rng = np.random.default_rng(SEMENTE)
SORTEIOS = 200_000

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
ROD_DEP = 'Fonte: Depner et al. 2019, Current Biology 29(6) · cálculo próprio | #365Probabilidades'
ROD_AK = 'Fonte: Åkerstedt et al. 2019, Journal of Sleep Research 28(1) · cálculo próprio | #365Probabilidades'

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def mil(x):
    return f"{x:,.0f}".replace(",", ".")

print("✅ Bibliotecas carregadas · semente", SEMENTE)


In [ ]:
# --- DADOS DA LITERATURA ---

# Depner et al. 2019 · 36 adultos, laboratório do sono
DEPNER = {'Sono curto o tempo todo': -13, 'Sono curto com recuperação no fim de semana': -27}
HORAS_SEMANA = 5          # oportunidade de sono por noite na semana
EXTRA_FDS = 1.0           # horas a mais por noite no fim de semana, em relação ao início
NOITES_FDS = 2

# Åkerstedt et al. 2019 · Tabelas 1 e 2 (coorte inteira, todas as idades)
COORTE = pd.DataFrame({
    'grupo': ['curto · curto', 'curto · médio ou longo', 'médio · médio', 'médio · longo',
              'longo · longo', 'longo · curto'],
    'semana_fds': ['≤5 h na semana e no fim de semana', '≤5 h na semana, mais no fim de semana',
                   '6 a 7 h nos dois', '6 a 7 h na semana, mais no fim de semana',
                   '≥8 h nos dois', '≥8 h na semana, menos no fim de semana'],
    'pessoas': [1592, 1438, 10699, 14381, 9174, 731],
    'mortes': [323, 85, 1302, 503, 918, 103],
    'pessoas_ano': [19310, 18356, 134434, 186408, 115555, 9078],
})
# Razões de risco ajustadas, menores de 65 anos (texto do artigo)
HR = {'curto · curto': (1.65, 1.22, 2.23), 'longo · longo': (1.25, 1.05, 1.50)}

print("=" * 72)
print("  DADOS DA LITERATURA")
print("=" * 72)
print("  Depner 2019 · queda da sensibilidade à insulina em duas semanas:")
for k, v in DEPNER.items():
    print(f"   {k:<46} {v}%")
print(f"\n  Åkerstedt 2019 · {mil(COORTE['pessoas'].sum())} analisados, {mil(COORTE['mortes'].sum())} mortes")
print(COORTE[['grupo', 'pessoas', 'mortes', 'pessoas_ano']].to_string(index=False))
print("=" * 72)


In [ ]:
# --- O MODELO ---

# CAMADA 1 · A contabilidade da dívida
print("=" * 72)
print("  CAMADA 1 · A CONTA DO SONO (premissas declaradas)")
print("=" * 72)
NECESSIDADES = [7, 7.5, 8]
DEVOLVE = EXTRA_FDS * NOITES_FDS
CONTA = {}
for nec in NECESSIDADES:
    divida = (nec - HORAS_SEMANA) * 5
    CONTA[nec] = (divida, DEVOLVE / divida)
    print(f"  Precisa de {vir(nec)} h · dívida da semana {vir(divida)} h · o fim de semana devolve"
          f" {vir(DEVOLVE)} h · paga {vir(DEVOLVE / divida * 100, 0)}% da dívida")

# CAMADA 2 · O metabolismo
print("\n" + "=" * 72)
print("  CAMADA 2 · O METABOLISMO (Depner 2019)")
print("=" * 72)
r = DEPNER['Sono curto com recuperação no fim de semana'] / DEPNER['Sono curto o tempo todo']
print(f"  A queda com recuperação no fim de semana foi {vir(r, 1)} vezes a queda de quem nunca recuperou")

# CAMADA 3 · As taxas de morte
print("\n" + "=" * 72)
print("  CAMADA 3 · MORTES POR 1.000 PESSOAS-ANO (taxas brutas, todas as idades)")
print("=" * 72)
POST = {}
for _, g in COORTE.iterrows():
    amostra = rng.gamma(g['mortes'] + 0.5, 1 / g['pessoas_ano'], SORTEIOS) * 1000
    POST[g['grupo']] = amostra
    lo, hi = np.percentile(amostra, [2.5, 97.5])
    print(f"  {g['grupo']:<24} {vir(g['mortes'] / g['pessoas_ano'] * 1000)}  IC 95% [{vir(lo)}; {vir(hi)}]")
P_COMP = (POST['curto · médio ou longo'] < POST['curto · curto']).mean()
RAZAO = POST['curto · médio ou longo'] / POST['curto · curto']
print(f"\n  P(quem compensa no fim de semana morre menos que quem nunca compensa): {vir(P_COMP * 100)}%")
print(f"  Razão bruta, compensa / nunca compensa: {vir(np.median(RAZAO), 2)}"
      f"  IC 95% [{vir(np.percentile(RAZAO, 2.5), 2)}; {vir(np.percentile(RAZAO, 97.5), 2)}]")
print("  Atenção: taxa bruta mistura idades. Quem compensa tende a ser mais jovem.")

# CAMADA 4 · As razões ajustadas
print("\n" + "=" * 72)
print("  CAMADA 4 · RAZÕES DE RISCO AJUSTADAS (menores de 65, publicadas)")
print("=" * 72)
for k, (h, lo, hi) in HR.items():
    se = (np.log(hi) - np.log(lo)) / (2 * 1.96)
    p_maior = stats.norm.sf(0, loc=np.log(h), scale=se)
    print(f"  {k:<24} HR {vir(h, 2)} [{vir(lo, 2)}; {vir(hi, 2)}]   P(HR > 1) = {vir(p_maior * 100)}%")
print("  curto · médio ou longo   sem diferença do grupo de 6 a 7 horas (número só no gráfico)")
print("=" * 72)


In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · A conta do sono
fig1, ax1 = plt.subplots(figsize=(11, 6))
x = np.arange(len(NECESSIDADES))
div = [CONTA[n][0] for n in NECESSIDADES]
ax1.bar(x, div, color=CINZA, width=0.55, label='dívida da semana')
ax1.bar(x, [DEVOLVE] * len(x), color=DOURADO, width=0.55, label='o que o fim de semana devolve')
for xi, n in zip(x, NECESSIDADES):
    ax1.text(xi, CONTA[n][0] + 0.3, f"paga {vir(CONTA[n][1] * 100, 0)}%", ha='center', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([f"precisa de {vir(n)} h" for n in NECESSIDADES], fontsize=12)
ax1.set_ylabel('Horas de sono na semana', fontsize=12)
ax1.legend(frameon=False, fontsize=11)
ax1.set_title('Cinco noites de 5 horas abrem uma dívida que o sábado e o domingo não fecham\n'
              'conta própria · cerca de 1 hora a mais por noite no fim de semana (Depner 2019)', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, ROD_DEP, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-156-grafico-01-a-conta-do-sono.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · O metabolismo
fig2, ax2 = plt.subplots(figsize=(11, 6))
nomes = list(DEPNER)
vals = [DEPNER[k] for k in nomes]
ax2.bar([0, 1], vals, color=[CINZA, VERMELHO], width=0.5)
for i, v in enumerate(vals):
    ax2.text(i, v / 2, f"{v}%", ha='center', va='center', fontsize=20, fontweight='bold', color='white')
ax2.axhline(0, color='#333', linewidth=1)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(['sono curto\no tempo todo', 'sono curto com\nrecuperação no fim de semana'], fontsize=12)
ax2.set_ylabel('Mudança na sensibilidade à insulina (%)', fontsize=11)
ax2.set_ylim(-32, 3)
ax2.set_title('Quem recuperou no fim de semana perdeu o dobro de sensibilidade à insulina\n'
              '36 adultos, duas semanas num laboratório do sono', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, ROD_DEP, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-156-grafico-02-o-metabolismo.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · As taxas de morte
fig3, ax3 = plt.subplots(figsize=(12, 6.5))
ordem = list(COORTE['grupo'])
y = np.arange(len(ordem))[::-1]
for yi, g in zip(y, ordem):
    a = POST[g]
    lo, md, hi = np.percentile(a, [2.5, 50, 97.5])
    cor = VERDE if g == 'curto · médio ou longo' else VERMELHO if g == 'curto · curto' else CINZA
    ax3.plot([lo, hi], [yi, yi], color=cor, linewidth=4)
    ax3.plot(md, yi, 'o', color=cor, markersize=12)
    ax3.text(hi + 0.4, yi, f"{vir(md)}", va='center', fontsize=11)
ax3.set_yticks(y)
ax3.set_yticklabels(COORTE['semana_fds'], fontsize=10.5)
ax3.set_xlabel('Mortes por 1.000 pessoas por ano (taxa bruta, IC 95%)', fontsize=11)
ax3.set_title('Nas taxas brutas, quem dorme pouco a semana inteira fica no topo. Parte da diferença é idade\n'
              'coorte sueca, 38.015 pessoas, 13 anos · taxas brutas, sem ajuste', fontsize=12.5, pad=14)
plt.figtext(0.5, 0.01, ROD_AK, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-156-grafico-03-as-taxas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · As razões ajustadas
fig4, ax4 = plt.subplots(figsize=(11, 5.5))
itens = [('≤5 h na semana e no fim de semana', HR['curto · curto'], VERMELHO),
         ('≥8 h na semana e no fim de semana', HR['longo · longo'], DOURADO)]
for yi, (nome, (h, lo, hi), cor) in zip([1, 0], itens):
    ax4.plot([lo, hi], [yi, yi], color=cor, linewidth=4)
    ax4.plot(h, yi, 'o', color=cor, markersize=13)
    ax4.text(hi + 0.03, yi, f"{vir(h, 2)} [{vir(lo, 2)}; {vir(hi, 2)}]", va='center', fontsize=12)
ax4.plot(1, -1, 'o', color=VERDE, markersize=13)
ax4.text(1.03, -1, 'sem diferença (número só no gráfico do artigo)', va='center', fontsize=11, color=VERDE)
ax4.axvline(1, color='#333', linestyle='--', linewidth=1.5)
ax4.set_yticks([1, 0, -1])
ax4.set_yticklabels(['dorme pouco a semana toda', 'dorme muito a semana toda', 'pouco na semana,\nmais no fim de semana'], fontsize=11)
ax4.set_xscale('log')
ax4.set_xticks([0.8, 1, 1.25, 1.5, 2, 2.5])
ax4.set_xticklabels(['0,8', '1', '1,25', '1,5', '2', '2,5'])
ax4.set_xlim(0.75, 3.2)
ax4.xaxis.set_minor_formatter(plt.NullFormatter())
ax4.set_xlabel('Razão de risco de morte contra quem dorme 6 a 7 horas · menores de 65 · IC 95%', fontsize=11)
ax4.set_title('A assinatura estatística: o risco ajustado de quem dorme pouco a semana toda\n'
              'ajustado por sexo, IMC, escolaridade, fumo, álcool, atividade, turnos e doenças', fontsize=13, pad=14)
plt.figtext(0.5, 0.01, ROD_AK, ha='center', fontsize=8.5, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-156-grafico-04-o-risco-ajustado.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")


### 💡 O Insight

**Para o metabolismo, dormir até tarde no sábado não compensou. No experimento, quem recuperou
no fim de semana perdeu 27% da sensibilidade à insulina, o dobro de quem nunca recuperou. Na
conta das horas, o fim de semana devolve só uma parte pequena da dívida, entre 13% e 20%.**

Mas a resposta muda quando a conta é outra. Numa coorte sueca de 13 anos, entre os menores de
65, quem dormia 5 horas ou menos a semana toda teve 65% mais risco de morrer do que quem dormia
6 a 7 horas. Quem dormia pouco na semana e mais no fim de semana não teve risco maior.

O sábado não paga a semana. No máximo, impede que a dívida fique impagável.

*Quantas horas o seu sábado está tentando devolver, e quantas a sua semana tirou?*

---

### ⚠️ Limitações do Modelo

- O experimento tem 36 pessoas jovens e saudáveis, por duas semanas. Os números do metabolismo
  vêm do resumo e do comunicado do NIH, sem os desvios por grupo, e por isso entram sem
  intervalo.
- A conta da dívida usa premissas declaradas (precisar de 7 a 8 horas, 5 noites de 5 horas,
  1 hora a mais por noite no fim de semana). É uma ilustração, não uma medida.
- As taxas da camada 3 são brutas e misturam todas as idades. Quem compensa no fim de semana
  tende a ser mais jovem, o que baixa a taxa sem dizer nada sobre o sono. A comparação justa é a
  da camada 4, ajustada e só para menores de 65.
- A coorte sueca mede o sono por autorrelato, uma única vez, em 1997. É associação, não causa.
- O artigo sueco dá o número exato do grupo que compensa no fim de semana só num gráfico.
- O jet lag social (Roenneberg 2012) entra como conceito, sem número no modelo.
- Sem ×0,80, porque são medidas de laboratório e mortes registradas.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
